# Kiểm thử lưu video và metadata (P3)

Notebook kiểm tra quy tắc tên file `event_yyyyMMdd_HHmmss.mp4`, đọc được video MP4 mẫu và JSON metadata có đủ trường. Đây là kiểm thử định dạng và luồng lưu; Colab không điều khiển được camera Android, nên ghi camera thật cần chạy ứng dụng trên điện thoại.

Mở notebook trong Google Colab: tải file này lên Colab hoặc đặt trong Google Drive rồi chọn **Open with → Google Colaboratory**.

In [ ]:
import json
import re
import tempfile
import time
from datetime import datetime
from pathlib import Path

import cv2
import numpy as np

thu_muc = Path(tempfile.mkdtemp(prefix="kiem_thu_p3_"))
moc_ten = datetime.now().strftime("%Y%m%d_%H%M%S")
tep_video = thu_muc / f"event_{moc_ten}.mp4"
tep_json = tep_video.with_suffix(".json")
print("Thư mục kiểm thử:", thu_muc)

In [ ]:
# Tạo clip thử ngắn để xác nhận MP4 được ghi và đọc lại được.
rong, cao, fps = 320, 240, 15
ghi = cv2.VideoWriter(str(tep_video), cv2.VideoWriter_fourcc(*"mp4v"), fps, (rong, cao))
assert ghi.isOpened(), "Không mở được bộ ghi MP4 trong môi trường Colab"
bat_dau_ms = int(time.time() * 1000)
for i in range(45):
    khung = np.zeros((cao, rong, 3), dtype=np.uint8)
    cv2.putText(khung, f"Khung {i + 1}", (70, 120), cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 255, 255), 2)
    ghi.write(khung)
ghi.release()
ket_thuc_ms = int(time.time() * 1000)
assert tep_video.exists() and tep_video.stat().st_size > 0, "Video không được lưu"

metadata = {
    "tep_video": tep_video.name,
    "bat_dau_ms": bat_dau_ms,
    "ket_thuc_ms": ket_thuc_ms,
    "thoi_luong_ms": max(0, ket_thuc_ms - bat_dau_ms),
    "kich_thuoc_byte": tep_video.stat().st_size,
}
tep_json.write_text(json.dumps(metadata, ensure_ascii=False, indent=2), encoding="utf-8")
print("Đã tạo video:", tep_video.name, "| bytes:", tep_video.stat().st_size)

In [ ]:
# Kiểm tra đúng mẫu tên, metadata trùng tên và MP4 có thể mở/đọc.
assert re.fullmatch(r"event_\d{8}_\d{6}\.mp4", tep_video.name), "Tên video không đúng mẫu"
doc = json.loads(tep_json.read_text(encoding="utf-8"))
assert doc["tep_video"] == tep_video.name
assert doc["ket_thuc_ms"] >= doc["bat_dau_ms"]
assert doc["kich_thuoc_byte"] == tep_video.stat().st_size
assert all(k in doc for k in ("bat_dau_ms", "ket_thuc_ms", "thoi_luong_ms", "kich_thuoc_byte"))
doc_video = cv2.VideoCapture(str(tep_video))
doc_duoc, khung_dau = doc_video.read()
so_khung = int(doc_video.get(cv2.CAP_PROP_FRAME_COUNT))
doc_video.release()
assert doc_duoc and khung_dau is not None, "Không đọc được khung hình từ MP4"
assert so_khung == 45, f"Số khung không đúng: {so_khung}"
print("ĐẠT: tên file, JSON metadata và video MP4 đều hợp lệ")
print(json.dumps(doc, ensure_ascii=False, indent=2))